# Notebook 2 — CNN: ครบ 10 เงื่อนไข

รันหลักบน Colab GPU และเก็บผลบน Google Drive อ่าน RUN_GUIDE.md ก่อนเริ่ม

Classification: Infiltration-only vs No Finding-only; official test หลังกรองครบ 12,081 ภาพ
XAI: bbox Infiltration 123 ภาพเดิม รวม mixed cases; คนไข้ไม่ซ้ำกับชุดฝึก

Baseline, Median L1–L3, CLAHE+DWT L1–L3, DAE+CLAHE L1–L3; ค่าทั้งหมดอยู่ใน shared/config.json

CLAHE+DWT เป็น joint presets (clip/depth/threshold); DAE+CLAHE เป็นการเปลี่ยน contrast หลัง DAE

## 1. สภาพแวดล้อมและเส้นทาง
ปรับ NOTEBOOK_2_DIR และ DATA_ROOT ให้ตรงกับตำแหน่งบน Drive ใช้ OUTPUT_ROOT เดียวกันทั้งสาม notebook เลือก SMOKE ค่าเดียวกันเสมอ

In [ ]:
from pathlib import Path
import sys

# Copy the complete Notebook_2 folder to Drive before opening a notebook.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")

NOTEBOOK_2_DIR = Path("/content/drive/MyDrive/nih-chest-xrays/notebooks/Notebook_2") if IN_COLAB else next(
    p for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "Notebook_2", Path.cwd() / "notebooks/Notebook_2"]
    if (p / "shared/config.json").is_file()
)
# None uses the local NIH directory, or the Colab path in shared/config.json.
DATA_ROOT = None
OUTPUT_ROOT = None  # Uses output_subdirectory in shared/config.json; same in all notebooks.
SMOKE = False  # True uses separate smoke_runs/ output and tiny validation data.

if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "opencv-python-headless", "PyWavelets", "scikit-learn", "pandas", "matplotlib"])
sys.path.insert(0, str(NOTEBOOK_2_DIR))
# Optional local test dependencies; nothing here changes the machine's global packages.
if not IN_COLAB and (NOTEBOOK_2_DIR / ".runtime").is_dir():
    sys.path.insert(0, str(NOTEBOOK_2_DIR / ".runtime"))

import torch
from shared.experiment import Experiment, versions
from shared.data import prepare_manifests
from shared.reporting import refresh_reports

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IN_COLAB and DEVICE != "cuda":
    raise RuntimeError("Select Runtime > Change runtime type > GPU before running the experiment")
exp = Experiment.open(NOTEBOOK_2_DIR, DATA_ROOT, OUTPUT_ROOT, smoke=SMOKE)
print("Device:", DEVICE, "| Mode:", exp.config["run_mode"])
print("Data:", exp.data_root, "| Results:", exp.output_root)
print(versions())


## 2. ตรวจข้อมูลและใช้ manifests ร่วมกัน
สุ่ม train 100/คลาส และ validation 25/คลาสด้วย seed 42 แบบแยกคนไข้ ไม่มีการย้ายภาพระหว่าง official train_val/test

In [ ]:
manifests = prepare_manifests(exp)
for name, frame in manifests.items():
    print(name, "images:", len(frame), "patients:", frame.patient_id.nunique())
print("Patient/image leakage checks passed. Shared manifests are locked.")
display(manifests["xai"].risk_flag.value_counts().to_frame("n_images"))


## 3. ฝึก DAE
ฝึกเฉพาะ train subset ใช้ validation noise คงที่ เลือก checkpoint จาก validation loss แล้วใช้ DAE ตัวเดียวกันทั้ง L1–L3

In [ ]:
from shared.training import train_dae, train_condition
dae_path = train_dae(exp, manifests, DEVICE)
print(dae_path)

## 4. ฝึกและประเมิน CNN ครบ 10 เงื่อนไข
ทุก run ใช้ seed, ImageNet initialization, train/validation/test และ hyperparameters เดียวกัน ค่าที่เสร็จแล้วจะโหลดกลับ; งานที่ค้างจะเริ่มต่อจาก epoch ล่าสุด

In [ ]:
for condition_id in exp.condition_ids:
    metrics = train_condition(exp, manifests, condition_id, DEVICE)
    print(condition_id, {k: metrics[k] for k in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')})
    refresh_reports(exp)

## ผลเปรียบเทียบและ README
ตารางรวมและรูปเปรียบเทียบอยู่ใน comparison/ ของแต่ละเกณฑ์ CI ของ XAI ใช้ patient bootstrap 2,000 รอบ seed เดียวกัน รายงาน Overall/Pure/Mixed-high-risk/Mixed-low-risk; ผลที่ยังไม่รันระบุ not_run

In [ ]:
status = refresh_reports(exp)
display(status)
from IPython.display import Markdown, display
display(Markdown((exp.output_root / 'README.md').read_text(encoding='utf-8')))